# GX_02 — Preparing the Data (Data Engineering)

**The question this notebook answers:** *Is the data ready for analysis and modelling?*

### What this notebook is about
GX_01 showed that both files are complete and technically clean, and it listed a few things to deal with: columns we should not use, text that has to be turned into dates and numbers, a PIN code that should not be stored, and patterns (like the 36% of customers with a zero balance) worth turning into their own columns.

Here we act on those findings. Every change is a deliberate decision, explained before the code that makes it.

In this notebook we:
1. **Clean** both files into *silver* tables.
2. **Build** a *gold* customer table with new, more useful columns and a fixed train / validation / test split.
3. **Prepare** the steps that turn customer data into the numeric format a Spark model needs.
4. **Show why Spark**: run the same steps on 1 million rows and compare the time.

### What this notebook produces
| Table | What is in it | Used by |
|---|---|---|
| `silver_churn` | Cleaned customers: only the useful columns, correct types | gold table |
| `silver_transactions` | Cleaned transactions: real dates, numeric coordinates, no PIN code | graph analysis (Phase 6), streaming (Phase 7) |
| `gold_churn_features` | Customers + new columns + train/val/test label | every later notebook |
| `scaletest_churn_x100` | A synthetic 1-million-row copy, **only** for the speed test | nothing else |

### Reminder: the three layers
- **Bronze** — untouched copy of the files (made in GX_01)
- **Silver** — cleaned data (this notebook)
- **Gold** — data ready for analysis and models (this notebook)

Every table here is rebuilt from bronze each time the notebook runs. Nothing is edited by hand, so anyone in the team can re-run it and get exactly the same tables.

## Step 0 — Preparing the notebook

**What we do:** load our shared helper code from `src/` and Spark's machine learning tools.

**Why:** the cleaning and feature rules live in `src/`, not in this notebook, so that later notebooks (and the streaming part in Phase 7) apply *exactly* the same rules to new data. New helper files for this phase:
- `cleaning.py` — the bronze → silver cleaning rules
- `features.py` — the customer groups and the new columns
- `ml.py` — the train/val/test split and the model preparation steps
- `utils.py` — a stopwatch and the synthetic data for the speed test

In [ ]:
# Make the shared helpers in src/ importable (this notebook lives in notebooks/).
import os
import sys

REPO_ROOT = os.path.abspath("..")
if REPO_ROOT not in sys.path:
    sys.path.append(REPO_ROOT)

from pyspark.ml import Pipeline
from pyspark.sql import functions as F

from src.cleaning import clean_churn, clean_transactions
from src.config import (
    BRONZE_CHURN, BRONZE_TRANSACTIONS, EXPECTED_ROWS, GOLD_CHURN, SCALE_FACTOR,
    SCALE_TEST_CHURN, SEED, SILVER_CHURN, SILVER_TRANSACTIONS, SPLIT_RATIOS,
)
from src.features import build_gold_features
from src.ingest import write_table
from src.ml import (
    CATEGORICAL_FEATURES, NUMERIC_FEATURES, add_split_column,
    build_preprocessing_stages, vector_slot_names,
)
from src.quality import quality_report
from src.utils import replicate_with_noise, timer

print("Spark version:", spark.version)

**Result:** Spark is running and all helper code loaded. We start from the bronze tables made in GX_01.

## Step 1 — Cleaning the customer data (`silver_churn`)

**What we do:** take `bronze_churn` and keep only what the analysis needs, with fixed types.

**The decisions, and why:**

| Column | Decision | Reason |
|---|---|---|
| `RowNumber` | Drop | It is just the line number in the file. It says nothing about the customer, and a model could even pick up patterns from the file order. |
| `Surname` | Drop | Personal data, and it never helps predict churn. Not storing it is a privacy point. |
| `CustomerId` | Keep, **as a key only** | We need it to identify customers (e.g. to give the retention team a list), but it is never used as a model input. |
| `Geography`, `Gender` | Trim spaces | They are already clean. Trimming protects against a future file where " France" and "France" would become two different countries. |
| `HasCrCard`, `IsActiveMember`, `Exited` | Make them whole numbers (0/1) | They already are. Doing it explicitly makes the rule visible and catches it if a future file sends "Yes"/"No". |
| `_corrupt_record` and tracking columns | Drop | Rows that failed to load would be removed here. GX_01 found 0 such rows, so no customer is lost. |

In [ ]:
silver_churn = clean_churn(spark.table(BRONZE_CHURN))
write_table(silver_churn, SILVER_CHURN)

silver_churn = spark.table(SILVER_CHURN)
print(f"{SILVER_CHURN}: {silver_churn.count():,} rows, {len(silver_churn.columns)} columns")
silver_churn.printSchema()

**Result:** **10,000 rows**, the same as bronze, so no customer was lost. The table went from 17 columns to **12**: the 11 customer columns we need plus `CustomerId` as the key. `RowNumber`, `Surname` and the 3 technical columns are gone.

**What we do:** run the *same* quality function as in GX_01, now on the silver table, and stop the notebook if anything fails.

**Why:** cleaning code can introduce new problems (a wrong type, a column lost, a filter that removes too much). Checking again after cleaning catches that immediately, instead of in the model three notebooks later.

In [ ]:
silver_churn_quality = quality_report(
    silver_churn,
    dataset="silver_churn",
    expected_rows=EXPECTED_ROWS["churn"],
    key_col="CustomerId",
    ranges={"CreditScore": (300, 850), "Age": (18, 100), "Tenure": (0, 10), "NumOfProducts": (1, 4)},
    allowed_values={
        "Geography": ["France", "Germany", "Spain"],
        "Gender": ["Male", "Female"],
        "HasCrCard": [0, 1],
        "IsActiveMember": [0, 1],
        "Exited": [0, 1],
    },
    rate_cols=["Exited"],
)
display(silver_churn_quality.groupBy("status").count().orderBy("status"))

n_failed = silver_churn_quality.filter("status = 'FAIL'").count()
if n_failed > 0:
    display(silver_churn_quality.filter("status = 'FAIL'"))
    raise ValueError(f"{n_failed} check(s) failed on silver_churn - fix the cleaning step.")
print("silver_churn passed every hard check.")

**Result:** **28 checks passed, 0 failures**, plus 1 INFO line (the churn rate, still 20.4%). The cleaning changed the shape of the table, not its content.

## Step 2 — Cleaning the transaction data (`silver_transactions`)

**What we do:** turn the text columns that GX_01 kept "as is" into usable values.

**The decisions, and why:**

| Column | Decision | Reason |
|---|---|---|
| `timestamp_raw` (text) | Turn into a real date/time, `event_time` | Text like "2025-01-17 10:14:00" cannot be sorted or grouped by time. A real timestamp can, which Phase 7 (streaming) needs. |
| `geolocation_raw` (text) | Split into two numbers, `latitude` and `longitude` | Numbers can be compared and mapped; text cannot. |
| Hemisphere letters (N/S/E/W) | Ignore them, **trust the sign of the number** | GX_01 showed every row says "W", even for positive longitudes like Paris (2.35). The letters contradict the numbers, so we follow the numbers. |
| `pin_code` | **Drop** | A security code has no analytical use, and storing it is a privacy and security risk. |

In [ ]:
silver_transactions = clean_transactions(spark.table(BRONZE_TRANSACTIONS))
write_table(silver_transactions, SILVER_TRANSACTIONS)

silver_transactions = spark.table(SILVER_TRANSACTIONS)
print(f"{SILVER_TRANSACTIONS}: {silver_transactions.count():,} rows, {len(silver_transactions.columns)} columns")
silver_transactions.printSchema()

**Result:** **1,000 rows**, nothing lost. `event_time` is now a real timestamp and `latitude` / `longitude` are numbers. `pin_code` is no longer in the table.

**What we do:** check that the conversion actually worked. If a date or a location does not match the expected format, the conversion gives an empty value (`null`) instead of an error, so we have to count those explicitly.

In [ ]:
silver_transactions.createOrReplaceTempView("silver_transactions")

display(spark.sql('''
    SELECT
        COUNT(*)                                          AS rows,
        COUNT_IF(event_time IS NULL)                      AS unreadable_times,
        COUNT_IF(latitude IS NULL OR longitude IS NULL)   AS unreadable_locations,
        MIN(event_time)                                   AS first_transaction,
        MAX(event_time)                                   AS last_transaction,
        COUNT(DISTINCT event_time)                        AS distinct_minutes,
        COUNT(DISTINCT latitude)                          AS distinct_latitudes,
        COUNT(DISTINCT longitude)                         AS distinct_longitudes,
        COUNT_IF(longitude < 0)                           AS west_of_greenwich
    FROM silver_transactions
'''))

**Result:**
- **0 unreadable dates and 0 unreadable locations.** Every row was converted.
- Times still run from 10:01 to 11:00 on 17 January 2025, over **60 distinct minutes**. The times are already whole minutes, so we do not need a separate "minute" column (the plan had one); `event_time` already is the minute.
- **6 latitudes × 6 longitudes**, as in GX_01. **310 transactions** now have a negative longitude (west of Greenwich, i.e. the American cities); the other 690 are east of it, as their numbers say, instead of all being "West".

Cleaning fixed the *format*, but it cannot fix the *content*: the coordinates are still real cities mixed up at random. We keep them so the table is complete, but we will not draw location conclusions from them.

**What we do:** put the location text from bronze next to the parsed numbers in silver for a few transactions, to see the conversion with our own eyes.

In [ ]:
display(
    spark.table(BRONZE_TRANSACTIONS).select("transaction_id", "geolocation_raw")
    .join(silver_transactions.select("transaction_id", "latitude", "longitude"), "transaction_id")
    .orderBy("transaction_id")
    .limit(5)
)

**Result:** the numbers now follow their own sign, not the letter.
- `TXN1026668467` said "34.0522 N, 2.3522 W". It is now longitude **+2.3522** (east of Greenwich), as the number says.
- `TXN1038301181` said "-74.006 W". It stays **-74.006** (west), so nothing changes when the number and the letter agree.
- The same rows also show the random mixing: `TXN1026668467` pairs Los Angeles's latitude (34.05) with Paris's longitude (2.35), a point in northern Algeria rather than a real customer location.

## Step 3 — Building the gold customer table (`gold_churn_features`)

**What we do:** add new columns that describe customers better, and give every customer a fixed label: train, validation or test.

**Why new columns:** a model only sees the columns we give it. Some patterns are easier to see if we spell them out. Each new column comes from something we found in GX_01:

| New column | What it is | Why |
|---|---|---|
| `AgeGroup` | 18–30 / 31–40 / 41–50 / 51–60 / 60+ | Churn changes a lot with age, and groups are easy to explain to management |
| `TenureGroup` | 0–2 / 3–6 / 7–10 years | Easy grouping for charts |
| `BalanceGroup`, `CreditScoreGroup` | Bands of balance and credit score | Same groups in the charts (GX_03) and the segments (Phase 5) |
| `ProductsFlag` | 1 / 2 / 3+ products | Only 326 customers have 3 or 4 products, so they are grouped together |
| `ZeroBalance` | 1 if the balance is exactly 0 | 36% of customers have an empty account; they may behave differently |
| `BalanceToSalary` | Balance divided by yearly salary | How much of their income a customer keeps with the bank |
| `LogBalanceToSalary` | A compressed version of the same ratio | Explained in section 3.1 |
| `Engagement` | 0 to 3: active member + has a credit card + has 2 or more products | One number for "how tied is this customer to the bank" |

The group cut points are defined once in `src/features.py`, so the bands on the slides are exactly the bands the model and the segments use.

**Why a fixed train / validation / test label:**
- **Train (70%)** — the model learns from these customers.
- **Validation (15%)** — used to compare models and choose settings.
- **Test (15%)** — kept locked until the very end, to measure the final model honestly, on customers it has never seen.

We fix this split **once, here**, and store it in the table. That way the model (Phase 4), the segments (Phase 5) and the graph (Phase 6) all use the same test customers, and no notebook can accidentally train on them.

The split is **stratified**: churners and stayers are split separately, so all three parts have the same churn rate (20.4%). It is also **repeatable**: the order comes from a fixed calculation on the customer ID (a hash), not from a random draw, so it gives the same split on every run and every machine.

In [ ]:
gold = (
    build_gold_features(silver_churn)
    .transform(lambda df: add_split_column(df, SPLIT_RATIOS, SEED))
)
write_table(gold, GOLD_CHURN)

gold = spark.table(GOLD_CHURN).cache()   # small table, read several times below
gold.createOrReplaceTempView("gold")
print(f"{GOLD_CHURN}: {gold.count():,} rows, {len(gold.columns)} columns")

display(gold.select(
    "CustomerId", "Age", "AgeGroup", "Balance", "ZeroBalance", "NumOfProducts", "ProductsFlag",
    "IsActiveMember", "HasCrCard", "Engagement", "BalanceToSalary", "split", "Exited",
).limit(5))

**Result:** **10,000 rows, 22 columns**: the 12 silver columns, 9 new columns and the `split` label. You can check the new columns by hand in the preview. The second customer (age 66) has 2 products, is active and has no credit card, so `Engagement` = 1 (active) + 0 (no card) + 1 (2+ products) = **2**, and `AgeGroup` is "60+". Customers with a balance of 0 have `ZeroBalance` = 1 and a ratio of 0.

### 3.1 Do the new columns tell us anything?

**What we do:** compare the churn rate across the values of two new columns, using Spark SQL on the gold table.

**Why:** a new column is only worth keeping if it separates customers who leave from customers who stay. If the churn rate is the same for every value, the column adds nothing.

In [ ]:
display(spark.sql('''
    SELECT 'ZeroBalance' AS feature, CAST(ZeroBalance AS STRING) AS value,
           COUNT(*) AS customers, ROUND(AVG(Exited) * 100, 1) AS churn_rate_pct
    FROM gold GROUP BY ZeroBalance
    UNION ALL
    SELECT 'Engagement', CAST(Engagement AS STRING),
           COUNT(*), ROUND(AVG(Exited) * 100, 1)
    FROM gold GROUP BY Engagement
    ORDER BY feature, value
'''))

**Result:** both columns clearly separate the two groups.
- **ZeroBalance:** customers with an empty account churn at **13.8%**, customers with money in it at **24.1%**. Surprisingly, the empty accounts are the *loyal* ones. (GX_01 also showed no German customer has a zero balance, so this partly overlaps with the Germany effect; GX_03 looks at that.)
- **Engagement:** churn falls steadily as engagement rises: **34.5%** at 0, **29.3%** at 1, **15.9%** at 2, and **9.1%** at 3. Customers with all three ties to the bank leave almost four times less often than those with none.

Both are kept.

**What we do:** look at how the balance-to-salary ratio is spread out.

**Why:** GX_01 found 59 customers with a yearly salary under €1,000 (the lowest is €11.58). Dividing by such a tiny salary gives a huge ratio. We need to know how extreme it gets before giving it to a model.

In [ ]:
display(gold.select("BalanceToSalary", "LogBalanceToSalary").summary("min", "25%", "50%", "75%", "max", "mean", "stddev"))

**Result:** the ratio is extremely uneven. Half of the customers are below **0.75** and three quarters are below **1.51**, but the largest value is **10,615**, caused by those unrealistic salaries. Because of a few extreme customers, the average (3.9) is five times the middle value, and the spread (standard deviation 108) is huge.

**Decision:** a few extreme values like these can pull a model (especially a linear one) towards a handful of odd customers. So the model gets **`LogBalanceToSalary`**, the logarithm of (1 + ratio). It keeps the same order (a higher ratio stays higher), but compresses the extremes: the maximum drops to **9.3** and the standard deviation to **0.77**. The original ratio stays in the table for charts, because it is easier to explain.

### 3.2 Checking the train / validation / test split

**What we do:** count the customers and the churn rate in each part.

**Why:** if the parts had different churn rates, a model could look better or worse just because of the split, not because of what it learned.

In [ ]:
display(spark.sql('''
    SELECT split,
           COUNT(*)                        AS customers,
           SUM(Exited)                     AS churned,
           ROUND(AVG(Exited) * 100, 2)     AS churn_rate_pct,
           ROUND(COUNT(*) / 10000 * 100, 1) AS share_pct
    FROM gold
    GROUP BY split
    ORDER BY CASE split WHEN 'train' THEN 1 WHEN 'val' THEN 2 ELSE 3 END
'''))

**Result:** the split works as intended.

| Part | Customers | Churned | Churn rate |
|---|---|---|---|
| Train | 7,000 | 1,426 | 20.37% |
| Validation | 1,499 | 305 | 20.35% |
| Test | 1,501 | 306 | 20.39% |

The sizes are 70 / 15 / 15 and the churn rate is the same in all three parts (20.4%), so they are fair copies of the whole customer base.

## Step 4 — Model preparation steps (Spark ML Pipeline)

**What we do:** define the steps that turn a customer row into the single list of numbers that a Spark model needs, and test them.

**Why:** Spark's models cannot read text like "France", and they work best when all numbers are on a similar scale. Four steps, chained in a **Pipeline** (a fixed sequence that is applied the same way every time):

| Step | What it does | Example |
|---|---|---|
| 1. StringIndexer | Turns text categories into numbers | France → 0, Germany → 1, Spain → 2 |
| 2. OneHotEncoder | Turns that number into separate 0/1 columns | Germany → [0, 1, 0]. Otherwise a model would read "Spain = 2" as "twice Germany". |
| 3. VectorAssembler | Puts all inputs into one list of numbers | [0, 1, 0, 1, 0, 619, 42, ...] |
| 4. StandardScaler | Puts every input on the same scale (average 0, spread 1) | Without it, a balance of 100,000 would dominate an age of 42 |

**Which columns go in:**
- **Text:** `Geography`, `Gender`
- **Numbers:** credit score, age, tenure, balance, number of products, credit card, active member, salary, `ZeroBalance`, `LogBalanceToSalary`, `Engagement`
- **Not used:** `CustomerId` (just an identifier), the group labels (the model sees the exact values instead), and `Exited` (that is the answer we predict).

**The most important rule: learn the steps from the training customers only.** The scaler learns an average and a spread; the indexer learns which countries exist. If they learned from all 10,000 customers, information about the test customers would leak into the model, and our final test score would look better than it really is. So we **fit on train only**, then apply the result to everyone.

In Phase 4 the model is simply added as a fifth step at the end of this same Pipeline.

In [ ]:
preprocessing = Pipeline(stages=build_preprocessing_stages(CATEGORICAL_FEATURES, NUMERIC_FEATURES))

train = gold.filter("split = 'train'")
preprocessing_model = preprocessing.fit(train)        # learns ONLY from the 7,000 training customers
prepared = preprocessing_model.transform(gold)         # ...then applies the same rules to all 10,000

slot_names = vector_slot_names(prepared)
print(f"Each customer becomes a list of {len(slot_names)} numbers:")
for i, name in enumerate(slot_names):
    print(f"  {i:2d}  {name}")

**Result:** every customer is now a list of **16 numbers**: 3 for the country (France, Germany, Spain), 2 for gender, and the 11 number columns.

Why 3 country columns and not 2? We told the indexer to keep an extra "unknown country" slot, so that a new country appearing later (for example in the streaming data of Phase 7) does not crash the job. The encoder drops one slot by default, and the one it drops is that "unknown" slot, so all three real countries keep their own column.

**What we do:** look at what the steps learned from the training customers, and at one prepared customer.

In [ ]:
indexer_model = preprocessing_model.stages[0]
scaler_model = preprocessing_model.stages[-1]

print("Categories learned from train:", indexer_model.labelsArray)

learned = dict(zip(slot_names, scaler_model.mean.toArray()))
print("Averages learned from train:")
for col in ["CreditScore", "Age", "Balance", "EstimatedSalary"]:
    print(f"  {col:16s} {learned[col]:>12,.1f}")

display(prepared.select("CustomerId", "Geography", "Gender", "Age", "Balance", "features").limit(3))

**Result:**
- The indexer found the 3 countries and 2 genders, ordered from most to least common (France, Germany, Spain; Male, Female).
- The scaler learned, from the training customers only, an average credit score of about **651**, age about **39**, balance about **€77,000** and salary about **€100,000**.
- In the `features` column every value is now around 0: positive means above the training average, negative means below. For example, the 66-year-old customer gets an age value of **+2.6** (well above the average of 39), while the 35-year-old gets **-0.4** (a little below).

This prepared data is **not saved** as a table. Phase 4 rebuilds it together with the model, from the same function, so the preparation and the model are always saved and reused as one unit.

## Step 5 — Why Spark? A scalability test

**The client's question:** *"We have 10,000 customers. Why not just use Excel or a laptop script?"*

**Our answer:** this prototype is built for the real bank, which has millions of customers and years of transactions. To show that, we run **exactly the same steps** on a much bigger version of the data and measure the time.

**What we do:**
1. Make a **synthetic** copy of the customer table, 100 times larger (**1,000,000 rows**). Each copy gets a new ID and a slightly changed balance and salary (±1%), so the rows are not identical.
2. Run the full chain on both sizes: cleaning → gold columns → split → learn the preparation steps on train → prepare every customer.
3. Compare the times.

> ⚠️ **This copy is fake data.** It is used only to measure speed. No result, chart or number about customers ever comes from it.

**How Spark handles it:** Spark splits the data into pieces and processes the pieces in parallel. When the data grows, a bigger cluster can simply process more pieces at the same time, without changing a single line of our code. On Databricks Free Edition the compute is small and fixed, so here we only show that the same code handles 100× more data; on a paid cluster it would also run faster.

In [ ]:
def prepare_customers(bronze_customers):
    # The full Phase 2 chain, from raw customers to model-ready customers.
    gold_df = (
        build_gold_features(clean_churn(bronze_customers))
        .transform(lambda df: add_split_column(df, SPLIT_RATIOS, SEED))
    )
    model = Pipeline(stages=build_preprocessing_stages()).fit(gold_df.filter("split = 'train'"))
    return model.transform(gold_df)


bronze_customers = spark.table(BRONZE_CHURN)
big_bronze_customers = replicate_with_noise(bronze_customers, SCALE_FACTOR, SEED)
print(f"Original: {bronze_customers.count():,} rows   Synthetic copy: {big_bronze_customers.count():,} rows")

**Result:** the synthetic copy has **1,000,000 rows**, 100 times the original 10,000.

**What we do next:** time the full chain on both. We force Spark to really compute every row by sending the output to a "no-op" destination (`noop`): Spark does all the work but throws the result away, so we time the processing and not the saving.

**Why this matters:** Spark is *lazy*. Lines like `clean_churn(...)` only describe the work; nothing runs until a result is actually needed. Without forcing it, we would be timing nothing.

In [ ]:
timings = {}

with timer("10,000 customers", timings):
    prepare_customers(bronze_customers).write.format("noop").mode("overwrite").save()

with timer("1,000,000 customers (synthetic)", timings):
    prepare_customers(big_bronze_customers).write.format("noop").mode("overwrite").save()

display(spark.createDataFrame(list(timings.items()), ["run", "seconds"]))

small, big = timings.values()
print(f"Data grew {SCALE_FACTOR}x; processing time grew {big / small:.1f}x "
      f"({small:.1f}s -> {big:.1f}s), with no change to the code.")

**Result:** the printed line above gives the result of this run. How to read it:
- **The same code ran on 100× more data without any change.** No row limits, no memory errors, no rewriting.
- **The time grows much less than the data.** A large part of the time on small data is fixed overhead (starting the jobs, planning). With more data that overhead is shared over many more rows, so each row costs less.
- Exact times change from run to run on shared Free Edition compute, so we report the comparison, not the precise seconds.

**What this means for the client:** the pipeline is ready for their full customer base. Growing from thousands to millions of customers is a question of cluster size, not of rebuilding the solution.

### 5.1 Storing big data smartly: partitioning

**What we do:** save the 1-million-row gold table split into folders by country (`partitionBy("Geography")`), then ask for German customers only, and look at Spark's plan for that query with `explain()`.

**Why:** in a real bank, most questions concern one country or region at a time. If the data is stored in one folder per country, Spark can skip the other countries' folders entirely instead of reading everything and filtering afterwards. `explain()` shows us the plan Spark made, so we can check that it really does this.

In [ ]:
big_gold = (
    build_gold_features(clean_churn(big_bronze_customers))
    .transform(lambda df: add_split_column(df, SPLIT_RATIOS, SEED))
)
write_table(big_gold, SCALE_TEST_CHURN, partition_by=["Geography"])

germany_only = spark.table(SCALE_TEST_CHURN).filter(F.col("Geography") == "Germany")
print(f"German customers in the synthetic table: {germany_only.count():,}")
germany_only.select("CustomerId", "Age", "Balance", "Exited").explain()

**Result:** the query returns **250,900** German rows (the 2,509 real German customers × 100). In the printed plan, look at the line that reads the table (on Databricks its name can differ, e.g. a Photon scan): it contains **`PartitionFilters: [... (Geography = Germany)]`**. This means Spark decided, *before reading anything*, to open only the Germany folder and skip France and Spain, about **75% of the data never read**. It also lists only the 4 columns we asked for (`ReadSchema`), so the other columns are not read either.

On 1 million rows this saves seconds; on years of data for millions of customers it saves most of the cost of every query.

## Summary

### What we did
- **Silver customers** (`silver_churn`): dropped row number and surname, kept the customer ID as a key only, fixed the types. 10,000 customers, nobody lost, all checks passed again after cleaning.
- **Silver transactions** (`silver_transactions`): real timestamps, numeric coordinates (following the number, not the wrong "W" letter), PIN code removed. 1,000 transactions, every value converted.
- **Gold customers** (`gold_churn_features`): 9 new columns and a fixed, stratified, repeatable train / validation / test split (7,000 / 1,499 / 1,501, churn 20.4% in each).
- **Model preparation Pipeline**: 4 steps that turn each customer into 16 numbers, learned from the training customers only.
- **Scalability test**: the same chain ran on a synthetic 1-million-row copy with no code change, and partitioning by country let Spark skip about 75% of the data for a single-country query.

### What we learned
- **Engagement matters a lot:** churn falls from 34.5% (no ties to the bank) to 9.1% (all three ties).
- **Empty accounts are loyal:** customers with a zero balance churn at 13.8% vs 24.1%.
- **Salary is unreliable:** a few tiny salaries make the balance-to-salary ratio explode (up to 10,615), so the model gets a compressed (log) version.
- **Cleaning cannot fix the transaction file's content:** the format is now correct, but the locations are still random city mixes, the data still covers one hour, and it still cannot be linked to customers.

### Decisions for later notebooks
- Every later notebook reads **`gold_churn_features`**, never bronze or silver.
- Models use **`split`**: train on `train`, compare on `val`, open `test` only once, at the very end.
- Models start from `build_preprocessing_stages()` and add the model as the last step, so preparation and model are always saved together.

---

## Next: GX_03 — Exploring the data with Spark SQL

The gold table is ready, so the next notebook answers the client's first business question: **who leaves the bank, and where?**
- Churn rate by country, gender, age group, number of products and activity, using Spark SQL and the DataFrame API.
- Combinations of factors, for example country × zero balance, and age × activity.
- Statistical tests (correlation, chi-square) to check which differences are real and which could be chance.
- 3–4 clear findings, each with one chart, which become slides for management and ideas to test in the model (Phase 4).